# How to learn features for functional maps on volumetric (tetrahedral) meshes

In this notebook, we demonstrate how to use the **Volumetric DiffusionNet** to learn feature descriptors for shape matching on tetrahedral meshes.

This extends the surface-based Deep Functional Maps pipeline to work with volumetric (tetrahedral) meshes.

In [ ]:
import os

os.environ["GEOMSTATS_BACKEND"] = "pytorch"

import torch

from geomfum.convert import P2pFromFmConverter
from geomfum.dataset.torch import PairsDataset, ShapeDataset
from geomfum.descriptor.learned import FeatureExtractor
from geomfum.forward_functional_map import ForwardFunctionalMap
from geomfum.learning.losses import (
    BijectivityLoss,
    LaplacianCommutativityLoss,
    LossManager,
    OrthonormalityLoss,
    FmapDescriptorsSupervisionLoss
)
from geomfum.learning.models import FMNet, RobustFMNet
from geomfum.learning.trainer import DeepFunctionalMapTrainer

## 1. Loading Tetrahedral Meshes

We load two tetrahedral meshes from the `dummy_tet` dataset using `shape_type="tetmesh"`. The dataset contains two volumetric meshes in MEDIT `.mesh` format.

The `ShapeDataset` now supports tetrahedral meshes alongside triangle meshes and point clouds.

In [ ]:
DATASET_PATH = "../../../datasets/tet_faust/"

# Use a small k for this demo (2 shapes, ~38k vertices)
K = 200

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


train_shapes = ShapeDataset(
    DATASET_PATH+"/train_set/",
    shape_type="tetmesh",
    spectral=True,
    distances=False,
    correspondences=False,
    device=device,
    k=K,
)
train_dataset = PairsDataset(train_shapes, device=device)

test_shapes = ShapeDataset(
    DATASET_PATH+"/test_set/",
    shape_type="tetmesh",
    spectral=True,
    distances=False,
    correspondences=False,
    device=device,
    k=K,
)
test_dataset = PairsDataset(test_shapes, device=device)
print(f"Loaded {len(train_shapes)} shapes")
for fname, shape in train_shapes.shapes.items():
    print(f"  {fname}: {shape.n_vertices} vertices, {shape.n_tets} tets")

In [ ]:
DATASET_PATH+"/train_set/"

In [ ]:
train_dataset[0]

## 2. Building the Volumetric DiffusionNet + FMNet Model

We use `FeatureExtractor.from_registry(which="volumetric_diffusionnet")` to get the volumetric feature extractor. For this small demo we use 2 blocks and 64 hidden channels.

The volumetric DiffusionNet adapts the original DiffusionNet architecture:
- **Spectral diffusion** (learned time diffusion) works unchanged
- **Spatial gradient features** are extended from 2D tangent-plane to 3D volumetric gradients

In [ ]:
from geomfum.descriptor.spectral import WaveKernelSignature
feature_extractor = FeatureExtractor.from_registry(
    which="volumetric_diffusionnet",
    device=device,
    in_channels=128,
    out_channels=128,
    hidden_channels=128,
    n_block=4,
    descriptor = WaveKernelSignature(n_domain=128, k=200),
    
    k=K,
)

fmap_module = ForwardFunctionalMap(1e3, 1, True, fmap_shape=(30,30))

model = RobustFMNet(
    feature_extractor=feature_extractor,
    fmap_module=fmap_module,    
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Setting Up Unsupervised Losses

Since we have no ground-truth correspondences for this demo, we use purely unsupervised losses:
- **Orthonormality**: $C^T C \approx I$
- **Bijectivity**: $C_{12} C_{21} \approx I$
- **Laplacian commutativity**: $C$ should commute with the Laplacian eigenvalues

In [ ]:
losses = [
    OrthonormalityLoss(weight=1.0),
    BijectivityLoss(weight=1.0),
    #LaplacianCommutativityLoss(weight=1e-3),
    FmapDescriptorsSupervisionLoss(weight=1e-3),
]
loss_manager = LossManager(losses)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 4. Training

We train for a few epochs. With only 2 shapes this is a proof-of-concept demo showing that the pipeline works end-to-end on volumetric meshes.

In [ ]:
trainer = DeepFunctionalMapTrainer(
    model=model,
    train_loss_manager=loss_manager,
    val_loss_manager=loss_manager,
    train_set=train_dataset,
    val_set=test_dataset,  # use the test set for validation
    optimizer=optimizer,
    device=device,
    epochs=5,
)

trainer.train()

## 5. Examining Results

Let's run the model in eval mode on one pair and inspect the functional map. For near-isometric shapes the functional map should be close to a diagonal matrix.

In [ ]:
import matplotlib.pyplot as plt

model.eval()
pair = train_dataset[0]
with torch.no_grad():
    result = model(pair["source"]["shape"], pair["target"]["shape"])

fmap12 = result["fmap12"].cpu().numpy()

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
im = ax.imshow(fmap12, cmap="RdBu", vmin=-1, vmax=1)
ax.set_title("Learned Functional Map C12")
ax.set_xlabel("Source basis")
ax.set_ylabel("Target basis")
plt.colorbar(im)
plt.tight_layout()
plt.show()

print(f"Functional map shape: {fmap12.shape}")
print(f"Diagonal dominance: {abs(fmap12.diagonal()).mean():.4f}")